In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/07_rag/04_rag_retrieval.py


In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/08_integration/01_copilot_orchestrator.py

In [0]:
import json
import time
from datetime import datetime

print("=" * 70)
print("PHASE 16 — COPILOT EVALUATION")
print("=" * 70)

print("Evaluation started:", datetime.now())

In [0]:
required_functions = [
    "classify_question",
    "generate_sql_request",
    "validate_sql",
    "execute_sql",
    "generate_question_embedding",
    "retrieve_documents",
    "build_rag_context",
    "generate_rag_answer",
    "decompose_hybrid_question",
    "run_sql_route",
    "run_hybrid_route",
    "ask_copilot"
]

print("=" * 70)
print("DEPENDENCY CHECK")
print("=" * 70)

dependency_results = []

for function_name in required_functions:

    available = callable(
        globals().get(function_name)
    )

    dependency_results.append({
        "function": function_name,
        "status": "PASS" if available else "FAIL"
    })

    print(
        f"{'PASS' if available else 'FAIL'} - {function_name}"
    )

failed_dependencies = sum(
    1
    for item in dependency_results
    if item["status"] == "FAIL"
)

print()
print("Total dependencies:", len(dependency_results))
print("Failed dependencies:", failed_dependencies)

if failed_dependencies > 0:
    raise RuntimeError(
        "Evaluation cannot continue because "
        "one or more required functions are unavailable."
    )

print("Dependency check PASSED.")

In [0]:
evaluation_questions = [
    {
        "id": "EVAL001",
        "question": "Which region generated the highest revenue?",
        "expected_route": "sql"
    },
    {
        "id": "EVAL002",
        "question": "What is the discount policy?",
        "expected_route": "rag"
    },
    {
        "id": "EVAL003",
        "question": "Which region generated the highest revenue and what discount policy applies there?",
        "expected_route": "hybrid"
    }
]

print(
    json.dumps(
        evaluation_questions,
        indent=2
    )
)

In [0]:
evaluation_results = []

print("=" * 70)
print("PHASE 16 — COPILOT EVALUATION")
print("=" * 70)

for test_case in evaluation_questions:

    test_id = test_case["id"]
    question = test_case["question"]
    expected_route = test_case["expected_route"]

    print()
    print("-" * 70)
    print(test_id)
    print("Question:", question)
    print("Expected route:", expected_route)

    start_time = time.time()

    try:

        response = ask_copilot(
            question
        )

        execution_time_ms = round(
            (time.time() - start_time) * 1000,
            2
        )

        if response is None:

            evaluation_results.append({
                "evaluation_id": test_id,
                "question": question,
                "expected_route": expected_route,
                "actual_route": None,
                "success": False,
                "route_match": False,
                "execution_time_ms": execution_time_ms,
                "error": "ask_copilot returned None"
            })

            continue

        actual_route = response.get(
            "route"
        )

        success = bool(
            response.get(
                "success",
                False
            )
        )

        route_match = (
            actual_route == expected_route
        )

        evaluation_results.append({
            "evaluation_id": test_id,
            "question": question,
            "expected_route": expected_route,
            "actual_route": actual_route,
            "success": success,
            "route_match": route_match,
            "execution_time_ms": execution_time_ms,
            "error": response.get("error")
        })

        print("Actual route:", actual_route)
        print("Success:", success)
        print("Route match:", route_match)
        print(
            "Execution time:",
            execution_time_ms,
            "ms"
        )

    except Exception as e:

        execution_time_ms = round(
            (time.time() - start_time) * 1000,
            2
        )

        evaluation_results.append({
            "evaluation_id": test_id,
            "question": question,
            "expected_route": expected_route,
            "actual_route": None,
            "success": False,
            "route_match": False,
            "execution_time_ms": execution_time_ms,
            "error": f"{type(e).__name__}: {str(e)}"
        })

        print(
            "ERROR:",
            f"{type(e).__name__}: {str(e)}"
        )

In [0]:
print("=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)

for result in evaluation_results:

    print()
    print(
        result["evaluation_id"],
        "->",
        "PASS" if (
            result["success"]
            and result["route_match"]
        ) else "FAIL"
    )

    print(
        "Question:",
        result["question"]
    )

    print(
        "Expected:",
        result["expected_route"]
    )

    print(
        "Actual:",
        result["actual_route"]
    )

    print(
        "Error:",
        result["error"]
    )

In [0]:
total_tests = len(
    evaluation_results
)

successful_tests = sum(
    1
    for r in evaluation_results
    if r["success"]
)

route_matches = sum(
    1
    for r in evaluation_results
    if r["route_match"]
)

failed_tests = (
    total_tests
    - successful_tests
)

success_rate = (
    successful_tests / total_tests
    if total_tests
    else 0
)

route_accuracy = (
    route_matches / total_tests
    if total_tests
    else 0
)

print("=" * 70)
print("PHASE 16 METRICS")
print("=" * 70)

print("Total tests:", total_tests)
print("Successful tests:", successful_tests)
print("Failed tests:", failed_tests)

print(
    "Success rate:",
    round(success_rate * 100, 2),
    "%"
)

print(
    "Route accuracy:",
    round(route_accuracy * 100, 2),
    "%"
)

In [0]:
overall_status = (
    "PASS"
    if (
        total_tests > 0
        and successful_tests == total_tests
        and route_matches == total_tests
    )
    else "FAIL"
)

print("=" * 70)
print("PHASE 16 — FINAL STATUS")
print("=" * 70)

print("Overall status:", overall_status)

if overall_status == "PASS":
    print("✓ SQL route passed")
    print("✓ RAG route passed")
    print("✓ Hybrid route passed")
    print("✓ All routing tests passed")
else:
    print("✗ One or more evaluation tests failed")
    print("Review the results above before continuing.")